# Tech Addiction Prediction: LightGBM Model
In this notebook, we build a robust tree-based model using `LightGBM`. 
We will implement a 5-Fold Stratified Cross Validation, optimizing the model for the competition metric (`ROC-AUC`), and train a final model on the entire dataset to make our probability predictions.


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings
warnings.filterwarnings('ignore')


## 1. Data Loading
We load the data from the standard Kaggle input directory.


In [ ]:
TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e8/train.csv'
TEST_PATH = '/kaggle/input/competitions/playground-series-s6e8/test.csv'
SUBMISSION_PATH = '/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv'

print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SUBMISSION_PATH)

X = train_df.drop(['id', 'addicted_label'], axis=1)
y = train_df['addicted_label'].values

print(f"Train features shape: {X.shape}")
print(f"Test features shape: {test_df.drop(['id'], axis=1).shape}")


## 1.5 Feature Engineering
Based on our EDA, we engineer new highly predictive features and ratios.

In [ ]:
def engineer_features(train, test):
    train = train.copy()
    test = test.copy()
    
    for df in [train, test]:
        # Original V2 Features
        df['weekend_delta'] = df['weekend_screen_time'] - df['daily_screen_time_hours']
        df['social_media_prop'] = df['social_media_hours'] / (df['daily_screen_time_hours'] + 1e-5)
        df['unaccounted_screen_time'] = df['daily_screen_time_hours'] - (df['social_media_hours'] + df['gaming_hours'] + df['work_study_hours'])
        
        # NEW V5 Features
        # Ratios
        df['productivity_ratio'] = df['work_study_hours'] / (df['daily_screen_time_hours'] + 1e-5)
        df['entertainment_ratio'] = (df['social_media_hours'] + df['gaming_hours']) / (df['daily_screen_time_hours'] + 1e-5)
        df['screen_to_sleep_ratio'] = df['daily_screen_time_hours'] / (df['sleep_hours'] + 1e-5)
        
        # Engagement Intensity
        df['interaction_intensity'] = df['notifications_per_day'] * df['app_opens_per_day']
        
        # Flags & Binning
        df['sleep_deprived'] = (df['sleep_hours'] < 6.5).astype(int)
        df['age_group'] = pd.cut(df['age'], bins=[0, 20, 30, 40, 50, 100], labels=False)
        
    return train, test

print("Engineering Deep V5 features...")
X, test_df_eng = engineer_features(X, test_df.drop(['id'], axis=1))
print(f"New Train features shape: {X.shape}")


## 2. Preprocessing Pipeline
Trees handle missing values natively. We encode categorical variables for the model.


In [ ]:
categorical_nominal = ['gender', 'academic_work_impact']
categorical_ordinal = ['stress_level']

nominal_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

ordinal_transformer = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=[['Low', 'Medium', 'High']], handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('nom', nominal_transformer, categorical_nominal),
        ('ord', ordinal_transformer, categorical_ordinal)
    ],
    remainder='passthrough'
)

print("Applying preprocessor...")
X_processed = preprocessor.fit_transform(X)
X_test_processed = preprocessor.transform(test_df_eng)
print(f"Processed Train shape: {X_processed.shape}")


## 2.5 Optuna Hyperparameter Tuning
We use Optuna to find the best hyperparameters before training our final 5-fold models.

In [ ]:
import optuna
from sklearn.metrics import roc_auc_score

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'random_state': 42,
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100)
    }
    
    # Fast 3-Fold CV for tuning
    skf_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    oof_auc = []
    
    for train_idx, val_idx in skf_tune.split(X_processed, y):
        X_tr, X_va = X_processed[train_idx], X_processed[val_idx]
        y_tr, y_va = y[train_idx], y[val_idx]
        
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr, 
            eval_set=[(X_va, y_va)], 
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )
        preds = model.predict_proba(X_va)[:, 1]
        oof_auc.append(roc_auc_score(y_va, preds))
        
    return np.mean(oof_auc)

print("Starting Optuna tuning...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
print("Best params found:", study.best_params)


## 3. Stratified 5-Fold Cross Validation
We evaluate the model using 5 folds to generate Out-Of-Fold (OOF) predictions and use early stopping during training.


In [ ]:
print("Starting 5-Fold Stratified Cross-Validation for LightGBM...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
models = []

lgb_params = study.best_params.copy()
lgb_params['objective'] = 'binary'
lgb_params['metric'] = 'auc'
lgb_params['random_state'] = 42

for fold, (train_idx, val_idx) in enumerate(skf.split(X_processed, y)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")
    X_train, X_val = X_processed[train_idx], X_processed[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    
    callbacks = [lgb.early_stopping(stopping_rounds=100, verbose=False)]
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=callbacks
    )
    
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    models.append(model)

print("\nCross-Validation complete!")


### Metric Evaluation


In [ ]:
print("Evaluating OOF predictions...")
roc_auc = roc_auc_score(y, oof_preds)
pr_auc = average_precision_score(y, oof_preds)

print("-" * 30)
print("LightGBM Model (5-Fold OOF) Performance:")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")
print("-" * 30)


## 4. Inference and Submission
We can predict the unseen test set using an ensemble (average) of the 5 models trained on the folds.


In [ ]:
print("Predicting on test set using Fold models...")
test_preds_proba = np.zeros(len(X_test_processed))

for model in models:
    test_preds_proba += model.predict_proba(X_test_processed)[:, 1] / len(models)

# Create submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'addicted_label': test_preds_proba
})

submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
display(submission.head())
